In [ ]:
#analisi fatta in ambiente google drive

In [ ]:
# SI FA UNA ANALISI DI REGRESSIONE MULTIVARIANZA

In [1]:
import pandas as pd
import numpy as np


In [61]:
#libreria per l'analisi dei dati
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression as lr

In [53]:
#libreria di supporto per l'analisi

# per lo scaling delle variabili numeriche
from sklearn.preprocessing import StandardScaler
# per eseguire il trainin e il test e la valutazione del modello
from sklearn.model_selection import train_test_split as envtest

In [46]:
#librerie grafiche
from matplotlib import pyplot as pyplot
import seaborn as sns
sns.set_theme()

In [5]:
df_incidenti = pd.read_csv( ".//Analisi//Incidenti_flat.csv")

In [6]:
df_incidenti.head(4)

,Unnamed: 0,index,TIME_PERIOD,COD_RIP,DEN_RIP,COD_REG,DEN_REG,COD_UTS,DEN_UTS,PRO_COM,COMUNE,AREA_KMQ,POP_RES,POP_AL_KMQ,RESULT,RESULT_DESC,OBS_VALUE
0,0,0,2015,1,Nord-ovest,1,Piemonte,201,Torino,1001,Agliè,13.1462,2701,205.46,F,Feriti,9
1,1,1,2016,1,Nord-ovest,1,Piemonte,201,Torino,1001,Agliè,13.1462,2644,201.12,F,Feriti,8
2,2,2,2018,1,Nord-ovest,1,Piemonte,201,Torino,1001,Agliè,13.1462,2658,202.19,F,Feriti,1
3,3,3,2019,1,Nord-ovest,1,Piemonte,201,Torino,1001,Agliè,13.1462,2634,200.36,F,Feriti,5


In [8]:
df_incidenti.columns

Index(['Unnamed: 0', 'index', 'TIME_PERIOD', 'COD_RIP', 'DEN_RIP', 'COD_REG',
       'DEN_REG', 'COD_UTS', 'DEN_UTS', 'PRO_COM', 'COMUNE', 'AREA_KMQ',
       'POP_RES', 'POP_AL_KMQ', 'RESULT', 'RESULT_DESC', 'OBS_VALUE'],
      dtype='object')

In [14]:
#feature : variabili indipendenti:  'TIME_PERIOD','PRO_COM','POP_AL_KMQ', 'RESULT'
#   NB le altre sono relazionate tra loro per cui le derivo semplicemente sapendo una di quelle considerate come feature
#   NB COME FATTO IN SCRIPT PRECEDENTE LE VARIABILI IND. POP_RES E AREA_KMQ SONO STATE RIDOTTE A UNA SOLA POP_AL_KMQ CHE RACCOGLIE AMBEDUE LE INFORMAZIONI
#     RIDUCENDO GLI EFFETTI NEGATIVI DI INSERIRE PIù FEATURE E DOVE CORREGGERE IL RELATIVO EFFETTO - ES R QUADRO ADJUSTMENT
# variabile target: variabile dipendente: 'OBS_VALUE'
df_analisi = df_incidenti[['TIME_PERIOD','PRO_COM','POP_AL_KMQ', 'RESULT','OBS_VALUE']]

In [15]:
df_analisi.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 77736 entries, 0 to 77735
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   TIME_PERIOD  77736 non-null  int64  
 1   PRO_COM      77736 non-null  int64  
 2   POP_AL_KMQ   77736 non-null  float64
 3   RESULT       77736 non-null  object 
 4   OBS_VALUE    77736 non-null  int64  
dtypes: float64(1), int64(3), object(1)
memory usage: 3.0+ MB


In [18]:
df_analisi['RESULT'] = df_analisi['RESULT'].astype('string')

/tmp/ipykernel_3087/2194147801.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_analisi['RESULT'] = df_analisi['RESULT'].astype('string')


In [19]:
df_analisi.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 77736 entries, 0 to 77735
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   TIME_PERIOD  77736 non-null  int64  
 1   PRO_COM      77736 non-null  int64  
 2   POP_AL_KMQ   77736 non-null  float64
 3   RESULT       77736 non-null  string 
 4   OBS_VALUE    77736 non-null  int64  
dtypes: float64(1), int64(3), string(1)
memory usage: 3.0 MB


In [20]:
# LA COLONNA TESTUALE LA CONVERTO IN UNA FEATURE BOOLEAN
# SICCOME IL DOMINIO VALORI AMMESSO E' DI SOLO 2 VALORI CREO UNA SOLA COLONNA 0-1
# IN CUI DERIVO L'ALTRA QUANDO QUESTA E' 0 --> RIMUOVO LA DIPENDENZA TRA 2 COLONNE DATO CHE RIESCO A RICAVARE L'ALTRA
df_analisi['MORTI']=0

/tmp/ipykernel_3087/4177000675.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_analisi['MORTI']=0


In [22]:
df_analisi[df_analisi['RESULT']=='M'].head(3)

,TIME_PERIOD,PRO_COM,POP_AL_KMQ,RESULT,OBS_VALUE,MORTI
9,2023,1001,195.34,M,1,0
20,2023,1002,233.11,M,1,0
35,2018,1004,143.29,M,1,0


In [41]:
maschera_filtro= df_analisi['RESULT']=='M'
nr_morti= df_analisi[maschera_filtro]['MORTI'].agg('size')
nr_morti

16183

In [42]:
nr_feriti= df_analisi[~maschera_filtro]['MORTI'].agg('size')
nr_feriti

61553

In [29]:
nr_tot= df_analisi['RESULT'].agg('size')
nr_tot

77736

In [34]:
nr_tot_ver = nr_feriti + nr_morti
print(nr_tot_ver)

77736


In [39]:
df_analisi.loc[maschera_filtro,'MORTI']=1

In [40]:
df_analisi.groupby('MORTI').agg('size')

,0
MORTI,
0,61553
1,16183


In [48]:
#verifico eventuale correlazione tra le feature scelte :
# MATRICE DI CORRELAZIONE
df_analisi[[ 'TIME_PERIOD','PRO_COM','POP_AL_KMQ', 'MORTI']].corr()

,TIME_PERIOD,PRO_COM,POP_AL_KMQ,MORTI
TIME_PERIOD,1.000000,0.009938,-0.004448,-0.017618
PRO_COM,0.009938,1.000000,0.008188,0.032555
POP_AL_KMQ,-0.004448,0.008188,1.000000,0.078191
MORTI,-0.017618,0.032555,0.078191,1.000000


In [49]:
#non esiste una forte correlazione tra le feature, la più alta è quello dei morti con la popolazione al kmq come ci si aspetterebbe
#ma il valore è tanto inferiore a 0.6 quindi non la ritengo correlata
# QUINDI NON SI EVIDENZIA SITUAZIONE DI MULTICOLLINEARITA'
0.078191 * 100

7.8191

In [57]:
# definizione delle variabile dipendente - target
Y_Target = df_analisi['OBS_VALUE']

In [59]:
# definizione delle feature - variabili indipendenti
X_Feature = df_analisi[[ 'TIME_PERIOD','PRO_COM','POP_AL_KMQ', 'MORTI']]

In [62]:
# aggiungo la costante
X_Feature = sm.add_constant(data=X_Feature)

In [63]:
# DEFINISCO GLI AMBIENTI DI T
X_train, X_test, Y_train, Y_test = envtest(X_Feature, Y_Target, test_size=0.3,random_state=2020)

In [70]:
#eseguo lo scaling ambiente Training
std_scaler = StandardScaler()
X_train_scaling = std_scaler.fit_transform(X_train)

In [71]:
#eseguo lo scaling ambiente test
std_scaler = StandardScaler()
X_test_scaling = std_scaler.fit_transform(X_test)

In [72]:
# definizione del modello statistico MULTI LINEAR REGRESSION
model_mlr= sm.OLS(Y_train ,X_train_scaling)

In [73]:
#allenamento del modello
MLR_Risultato=model_mlr.fit()

In [74]:
#lettura statistiche E LETTURA DEI PARAMETRI INTERNI
MLR_Risultato.summary()

/usr/local/lib/python3.12/dist-packages/statsmodels/regression/linear_model.py:1966: RuntimeWarning: divide by zero encountered in scalar divide
  return np.sqrt(eigvals[0]/eigvals[-1])


<class 'statsmodels.iolib.summary.Summary'>
"""
                                 OLS Regression Results                                
=======================================================================================
Dep. Variable:              OBS_VALUE   R-squared (uncentered):                   0.038
Model:                            OLS   Adj. R-squared (uncentered):              0.038
Method:                 Least Squares   F-statistic:                              532.4
Date:                Sat, 16 May 2026   Prob (F-statistic):                        0.00
Time:                        08:53:37   Log-Likelihood:                     -3.7641e+05
No. Observations:               54415   AIC:                                  7.528e+05
Df Residuals:                   54411   BIC:                                  7.529e+05
Df Model:                           4                                                  
Covariance Type:            nonrobust                                                  
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const               0          0        nan        nan           0           0
x1            -1.8149      1.048     -1.732      0.083      -3.868       0.239
x2             1.5210      1.048      1.451      0.147      -0.533       3.575
x3            46.2413      1.050     44.034      0.000      44.183      48.300
x4           -17.5782      1.051    -16.728      0.000     -19.638     -15.519
==============================================================================
Omnibus:                   167984.140   Durbin-Watson:                   1.968
Prob(Omnibus):                  0.000   Jarque-Bera (JB):      18962017039.954
Skew:                          47.640   Prob(JB):                         0.00
Kurtosis:                    2893.365   Cond. No.                          inf
==============================================================================

Notes:
[1] R² is computed without centering (uncentered) since the model does not contain a constant.
[2] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[3] The smallest eigenvalue is      0. This might indicate that there are
strong multicollinearity problems or that the design matrix is singular.
"""

In [ ]:
# VALUTAZIONE DEL COEFFICIENTE DI DETERMINAZIONE
# mi serve a valutare il modello matematico scelto per eseguire la previsione statistica
# Valutazione R Quadro

# Valutazione R Quadro Adjustment

In [ ]:
# Valutazione P_Value

In [77]:
# CALCOLO INDICE DI PERFORMANCE
# MAE - MEAN ABSOLUTE ERROR - ERRORE ASSOLUTO MEDIO
# MAE_TEST
def mae(y_reali, y_predetti):
    return round(np.mean(np.abs(y_reali - y_predetti)), 2)

In [85]:
# eseguo le previsioni sui dati mai visti
pred_test = MLR_Risultato.predict(X_test_scaling)

In [89]:
# MAE PREVISIONALE SU DATI DI TEST
MAE_TEST_SORG =  mae(Y_test , np.mean(Y_test))

In [90]:
# MAE PREVISIONALE SU DATI DI TRAINING
MAE_TEST_PREV =  mae(Y_test, np.mean(pred_test))

In [91]:
print(f'MAE TEST SORGENTE       : {MAE_TEST_SORG}')
print(f'MAE TEST PREVISIONALE  : {MAE_TEST_PREV}')

MAE TEST SORGENTE       : 39.38
MAE TEST PREVISIONALE  : 29.98


In [76]:
# leggo il valore di predizione sul dato di training
pred_train = MLR_Risultato.predict(X_train_scaling)

In [86]:
# MAE PREVISIONALE SU DATI DI TRAINING
MAE_TRAIN_SORG =  mae(Y_train, np.mean(Y_train))

In [87]:
# MAE PREVISIONALE SU DATI DI TRAINING
MAE_TRAIN_PREV =  mae(Y_train, np.mean(pred_train))

In [88]:
print(f'MAE TRAINING SORGENTE       : {MAE_TRAIN_SORG}')
print(f'MAE TRAINING PREVISIONALE  : {MAE_TRAIN_PREV}')

MAE TRAINING SORGENTE       : 38.6
MAE TRAINING PREVISIONALE  : 29.48
